![Partners](../../images/partners.png)

# Build Your Own SFINCS Hydrodynamic Model

## Overview

This interactive notebook allows you to build a custom **SFINCS hydrodynamic model** for any location in East Africa. Using an interactive map, you can select your model region, configure model parameters, and generate a complete hydrodynamic model ready for simulation.

### What is HydroMT-SFINCS?

[HydroMT-SFINCS](https://deltares.github.io/hydromt_sfincs/latest/index.html) is a Python-based framework that automates the setup of SFINCS flood models. It handles the processing of global and regional datasets (elevation, land use, soil, forcing data) into the specific input format required by SFINCS. Using a single build command, it:
- Generates the computational grid and elevation
- Derives Manning's roughness from land use data
- Sets up infiltration parameters from soil data
- Configures boundary conditions and forcing
- Creates all required input files

### What You'll Do

1. **Select your model region** using an interactive map
2. **Define the model domain** as a GeoJSON polygon
3. **Build your model** automatically using HydroMT-SFINCS
4. **Configure forcing** (precipitation and discharge) for your simulation
5. **Visualize the results** to verify your model setup

### Prerequisites

Before starting, ensure you have:
- Completed the SFINCS training notebook (`1_Run_DireDawa_SFINCS_model.ipynb`)
- Selected the **"HydroMT SFINCS (pixi)"** kernel (top-right corner)
- Prepared your data catalogs and configuration files

---

## Part 1: Environment Setup and Verification

### 1.1 Select the Correct Kernel

**⚠️ IMPORTANT**: Before running any code, ensure you have selected the correct kernel:

1. Go to **Kernel** → **Change Kernel** in the Jupyter menu in the top right corner of this notebook
2. Select **"HydroMT SFINCS (pixi)"** or the kernel that contains your HydroMT-SFINCS installation

The kernel name should appear in the top-right corner of this notebook. If you see a different kernel name, click on it to change it.

### 1.2 Verify HydroMT-SFINCS Installation

Let's verify that HydroMT is properly installed and can recognize the SFINCS model plugin.

**Expected output**: You should see `sfincs` listed among the available model plugins.

In [ ]:
# List all available HydroMT model plugins
!hydromt --models

**Understanding the output**:
- `model` and `example_model`: Base HydroMT classes
- `sfincs`: The SFINCS hydrodynamic model plugin — this is what we need
- Other models may also appear depending on installed plugins

### 1.3 Import Required Libraries

In [ ]:
# Standard library imports
import os
import yaml
import subprocess
import shlex
from pathlib import Path
from datetime import datetime

# Data manipulation
import numpy as np
import pandas as pd
import xarray as xr

# Geospatial libraries
import geopandas as gpd
from shapely.geometry import Point, box, shape

# Visualization
import matplotlib.pyplot as plt
import folium
from folium import plugins

# Load model
from hydromt_sfincs import SfincsModel

print("✓ All imports successful!")

---

## Part 2: Interactive Map for Region Selection

### 2.1 Create Interactive Map

Use this interactive map to:
- **Visualize** reference layers (coastal areas, rivers, elevation)
- **Click** to define your model region bounding box
- **Draw** a polygon to define your exact model domain
- **View** the coordinates of your selected area

The map will display your click coordinates in latitude and longitude.

**Tip**: Use the rectangle or polygon drawing tool (icons on the left side of the map) to define your model region. Copy the coordinates from the drawn shape to use in the next step.

In [ ]:
# Initialize coordinate storage
selected_coords = {'lat': None, 'lon': None, 'bbox': None}

def create_interactive_map(
    center=[9.5, 41.0], 
    zoom=7, 
    geojson_files=None):
    """
    Create an interactive Folium map with click functionality for SFINCS model setup.
    
    Parameters:
    -----------
    center : list
        [latitude, longitude] for map center (default: Dire Dawa region)
    zoom : int
        Initial zoom level
    geojson_files : dict
        Dictionary of {layer_name: file_path} for GeoJSON layers
    
    Returns:
    --------
    folium.Map
        Interactive map object
    """
    # Create base map
    m = folium.Map(
        location=center,
        zoom_start=zoom,
        tiles='OpenStreetMap',
        control_scale=True
    )
    
    # Add alternative base layers
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Satellite',
        overlay=False,
        control=True
    ).add_to(m)
    
    # Add satellite imagery with labels overlay
    folium.TileLayer(
        tiles='https://server.arcgisonline.com/ArcGIS/rest/services/Reference/World_Boundaries_and_Places/MapServer/tile/{z}/{y}/{x}',
        attr='Esri',
        name='Labels',
        overlay=True,
        control=True,
        show=True
    ).add_to(m)
    
    # Add GeoJSON layers if provided
    if geojson_files:
        colors = ['blue', 'green', 'red', 'purple', 'orange']
        for idx, (layer_name, file_path) in enumerate(geojson_files.items()):
            if Path(file_path).exists():
                try:
                    color = colors[idx % len(colors)]
                    folium.GeoJson(
                        str(file_path),
                        name=layer_name,
                        style_function=lambda x, c=color: {
                            'fillColor': c,
                            'color': c,
                            'weight': 2,
                            'fillOpacity': 0.3
                        }
                    ).add_to(m)
                except Exception as e:
                    print(f"Warning: Could not load {layer_name}: {e}")
    
    # Add click functionality for coordinate capture
    m.add_child(folium.LatLngPopup())
    
    # Add drawing tools for bounding box
    draw = plugins.Draw(
        export=False,
        draw_options={
            'polyline': False,
            'rectangle': True,
            'polygon': True,
            'circle': False,
            'circlemarker': False,
            'marker': True
        }
    )
    draw.add_to(m)
    
    # Add measure control
    plugins.MeasureControl(position='topleft', primary_length_unit='kilometers').add_to(m)
    
    # Add layer control
    folium.LayerControl().add_to(m)
    
    # Add custom HTML for instructions
    legend_html = '''
    <div style="position: fixed; 
                top: 10px; right: 10px; width: 280px; height: auto; 
                background-color: white; z-index:9999; font-size:14px;
                border:2px solid grey; border-radius: 5px; padding: 10px">
    <h4 style="margin-top: 0;">SFINCS Model Setup</h4>
    <p><b>1.</b> Use the rectangle tool to define your model domain</p>
    <p><b>2.</b> Click on the map to see coordinates</p>
    <p><b>3.</b> Copy the bounding box coordinates</p>
    <p><b>4.</b> Enter them in the cell below</p>
    </div>
    '''
    m.get_root().html.add_child(folium.Element(legend_html))
    
    return m

# Create and display the map
print("Creating interactive map...")
map_widget = create_interactive_map(
    center=[9.5, 41.0],  # Default center (Dire Dawa region)
    zoom=7,
)

display(map_widget)

### 2.2 Define the Model Region

After selecting your area on the map above, define the model region as a GeoJSON polygon. The cell below contains a pre-defined polygon for the Dire Dawa region as an example — **replace the coordinates with your own selection**.

**Coordinate Order**: GeoJSON uses Longitude (X), Latitude (Y) order.

In [ ]:
shp = input("Copy-Paste shapefile information: ")

In [ ]:
# Convert in to shapefile
shp = eval(shp)
polygon = shape(shp['geometry'])
gdf = gpd.GeoDataFrame({'geometry': [polygon]}, crs="EPSG:4326")

# Visualize the selected region
gdf.plot(edgecolor='red', facecolor='lightblue', alpha=0.5, figsize=(8, 6))
plt.title("Selected Model Region", fontsize=12, fontweight="bold")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"✓ Region defined with {len(polygon.exterior.coords)} vertices")
print(f"  Bounding box: {polygon.bounds}")

### 2.3 Save Region to GeoJSON

Save the selected region as a GeoJSON file that HydroMT will use as the `--region` argument during the build process.

In [ ]:
# Save the region as a GeoJSON file
gdf.to_file("region.geojson", driver='GeoJSON')
print("✓ Region saved to region.geojson")

---

## Part 3: Building the SFINCS Model

### 3.1 Understanding the `hydromt build` Command

The `hydromt build` command is the primary tool for creating new models. For SFINCS, its general syntax is:

```bash
hydromt build sfincs MODEL_ROOT --region REGION -i CONFIG_FILE -d DATA_CATALOG [options]
```

**Key arguments**:
- `sfincs`: We're building a SFINCS hydrodynamic model
- `MODEL_ROOT`: Output directory where the model will be saved
- `--region`: Defines the geographic extent (as a GeoJSON file, bounding box, or basin definition)
- `-i / --config`: Path to the build configuration YAML file
- `-d / --data`: Path to the data catalog YAML file
- `--force-overwrite`: Overwrite existing model files
- `-v` / `-vv`: Verbosity level for logging

### 3.2 Execute the Build Command

Now we'll build the SFINCS model using HydroMT. This process:
1. Reads the region definition from `region.geojson`
2. Creates the computational grid based on the configuration
3. Downloads and processes elevation data (DEM)
4. Derives Manning's roughness from land use data
5. Sets up infiltration parameters from soil data
6. Generates all required SFINCS input files

**Note**: The build can take several minutes depending on the region size, grid resolution, and internet connection speed.

In [ ]:
# Set some paths
# Resolve the data root relative to this notebook's location
notebook_dir = os.getcwd()
data_root = os.path.abspath(os.path.join(notebook_dir, "../../../data/hydroMT_data/hydromt_east_africa"))

# Load the original catalog
catalog_path = "./model_build_files/hydromt_east_africa_sfincs.yml"
with open(catalog_path, "r") as f:
    catalog = yaml.safe_load(f)

# Update the root path
catalog["root"] = data_root + os.sep

# Save a local copy with the correct root
local_catalog = "./hydromt_east_africa_sfincs_local.yml"
with open(local_catalog, "w") as f:
    yaml.dump(catalog, f, default_flow_style=False)

print(f"Data root set to: {data_root}")
print(f"Local catalog saved to: {local_catalog}")

In [ ]:
# Execute the HydroMT SFINCS build
model_name = input("Enter model name: ")
model_output_dir = f"./sfincs_models/{model_name}"
config_path = "./model_build_files/sfincs_build_my_own_model.yml"
data_catalog_path = "./hydromt_east_africa_sfincs_local.yml"
region = "{'geom': 'region.geojson'}"

print(f"Building SFINCS model: {model_name}")
print("=" * 60)
print(f"Output directory: {model_output_dir}")
print(f"Build config: {config_path}")
print(f"Data catalog: {data_catalog_path}")
print(f"Region: {region}")
print("\nExecuting HydroMT build...\n")

cmd = (
    f"hydromt build sfincs {model_output_dir} "
    f"--region \"{region}\" "
    f"-i {config_path} "
    f"-d {data_catalog_path} "
    "--fo -vv"
)

process = subprocess.Popen(
    shlex.split(cmd),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

process.wait()

print("\n" + "=" * 60)
print("Model build complete.")
print(f"Model location: {model_output_dir}")

**Command breakdown**:
- `sfincs`: We're building a SFINCS model
- `my_own_sfincs_model`: Output directory for the model
- `--region "{'geom': 'region.geojson'}"`: Use the GeoJSON file we saved as the model domain
- `-i ../model_build_files/sfincs_build_my_own_model.yml`: Build configuration file defining data sources and processing steps
- `--force-overwrite`: Replace any existing model with the same name
- `-v`: Verbose logging to track progress
- `-d ../model_build_files/hydromt_east_africa_sfincs.yml`: Data catalog specifying where to find elevation, land use, soil, and forcing data

**What happens during the build?**

The build process executes several steps defined in the configuration YAML:

1. **Grid setup** (`setup_grid`): Creates the computational grid at the specified resolution
2. **Elevation** (`setup_dep`): Processes the DEM and interpolates to the grid
3. **Sub-grid tables** (`setup_subgrid`): Creates sub-grid lookup tables for improved accuracy
4. **Roughness** (`setup_manning_roughness`): Derives Manning's n from land use classification
5. **Infiltration** (`setup_cn_infiltration`): Computes curve numbers from soil and land use data
6. **Mask** (`setup_mask_active`): Defines the active computational cells

---

## Part 4: Static Forcing Configuration

### 4.1 Overview

Configure static (constant) forcing values for your model. This is useful for testing or when dynamic forcing data is not available.

SFINCS requires two main types of forcing:
- **Precipitation**: Rainfall intensity driving pluvial flooding (mm/hr)
- **Discharge**: River inflow at source points driving fluvial flooding (m³/s)

**Note**: These values will be constant throughout the simulation period. For realistic simulations, you would replace these with time-varying forcing from observed or forecasted data.

In [ ]:
# ============================================
# FORCING CONFIGURATION
# ============================================
print("Precipitation Forcing Configuration")
print("=" * 60)

print("\nChoose forcing type:")
print("  1 = Static (constant rainfall)")
print("  2 = Design storm (realistic time-varying hyetograph)")
forcing_type = input("\nForcing type [1 or 2]: ") or "2"

# Simulation period
print("\nSimulation Period:")
start_date_str = input("  Start Date (YYYYMMDD HHMMSS) [20200101 000000]: ") or "20200101 000000"
end_date_str   = input("  End Date   (YYYYMMDD HHMMSS) [20200103 000000]: ") or "20200103 000000"
start_date = pd.to_datetime(start_date_str, format="%Y%m%d %H%M%S")
end_date   = pd.to_datetime(end_date_str,   format="%Y%m%d %H%M%S")

# Output timestep
dtout = float(input("  Output time step [seconds] [3600]: ") or "3600")

# Create hourly time array
times = pd.date_range(start_date, end_date, freq="1h")
precip_ts = np.zeros(len(times))

if forcing_type == "1":
    # ---- STATIC FORCING ----
    print("\nStatic Precipitation:")
    precip_value = float(input("  Constant precipitation [mm/hr] [20]: ") or "20")
    precip_ts[:] = precip_value
    storm_label = f"Static: {precip_value:.0f} mm/hr"

else:
    # ---- DESIGN STORM ----
    print("\nDesign Storm Configuration:")
    print("  Example scenarios for East Africa:")
    print("  ┌──────────────────────┬────────┬──────────┬──────┐")
    print("  │ Scenario             │ Total  │ Duration │ Peak │")
    print("  ├──────────────────────┼────────┼──────────┼──────┤")
    print("  │ Moderate event       │  80 mm │  4 hrs   │ 0.4  │")
    print("  │ Severe flash flood   │ 150 mm │  6 hrs   │ 0.3  │")
    print("  │ Extended storm       │ 200 mm │ 12 hrs   │ 0.5  │")
    print("  └──────────────────────┴────────┴──────────┴──────┘")

    total_rainfall = float(input("\n  Total rainfall [mm] [150]: ") or "150")
    storm_duration = float(input("  Storm duration [hours] [6]: ") or "6")
    peak_position  = float(input("  Peak position (0=start, 0.5=middle, 1=end) [0.4]: ") or "0.4")
    storm_start_hr = float(input("  Hours after sim start before storm begins [6]: ") or "6")

    # Build storm shape using beta distribution
    try:
        from scipy.stats import beta
        storm_steps = int(storm_duration)
        t_storm = np.linspace(0, 1, storm_steps)
        a_param = peak_position * 5 + 1
        b_param = (1 - peak_position) * 5 + 1
        storm_shape = beta.pdf(t_storm, a_param, b_param)
    except ImportError:
        # Fallback: triangular storm profile (no scipy needed)
        print("  (scipy not found — using triangular storm profile)")
        storm_steps = int(storm_duration)
        peak_idx = int(peak_position * storm_steps)
        storm_shape = np.zeros(storm_steps)
        if peak_idx > 0:
            storm_shape[:peak_idx] = np.linspace(0, 1, peak_idx)
        if peak_idx < storm_steps:
            storm_shape[peak_idx:] = np.linspace(1, 0, storm_steps - peak_idx)

    # Normalize so total = total_rainfall
    storm_intensity = storm_shape / storm_shape.sum() * total_rainfall

    # Place storm in timeline
    storm_start_idx = int(storm_start_hr)
    storm_end_idx = min(storm_start_idx + storm_steps, len(precip_ts))
    actual_steps = storm_end_idx - storm_start_idx
    precip_ts[storm_start_idx:storm_end_idx] = storm_intensity[:actual_steps]

    storm_label = f"Design storm: {total_rainfall:.0f} mm / {storm_duration:.0f} hrs"


In [ ]:
# ============================================
# PLOT THE HYETOGRAPH
# ============================================
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), gridspec_kw={'height_ratios': [2, 1]})

ax1.bar(times, precip_ts, width=pd.Timedelta(hours=0.9), color='steelblue', alpha=0.8)
ax1.set_ylabel('Precipitation [mm/hr]')
ax1.set_title(f'{storm_label}\nPeak intensity: {precip_ts.max():.1f} mm/hr')
ax1.grid(True, alpha=0.3)

cumulative = np.cumsum(precip_ts)
ax2.plot(times, cumulative, color='darkblue', linewidth=2)
ax2.fill_between(times, cumulative, alpha=0.2, color='steelblue')
ax2.set_ylabel('Cumulative [mm]')
ax2.set_xlabel('Time')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nForcing summary:")
print(f"  Total rainfall:   {cumulative[-1]:.1f} mm")
print(f"  Peak intensity:   {precip_ts.max():.1f} mm/hr")
print(f"  Simulation:       {start_date} → {end_date}")

In [ ]:
mod = SfincsModel(f"./sfincs_models/{model_name}", mode="r")
model_path = Path(f"./sfincs_models/{model_name}")
print(f"Model region: {mod.region}")
print(f"Model CRS:    {mod.crs}")

In [ ]:
# ============================================
# WRITE FORCING FILES TO MODEL
# ============================================

print("Writing forcing files...")
print("=" * 60)

# ---- 1. Write precipitation as simple ASCII time series ----
# This is SFINCS's native format for uniform precipitation
model_path = Path(model_path)

precip_path = model_path / "sfincs.precip"
tref = start_date

with open(precip_path, 'w') as f:
    for i, t in enumerate(times):
        seconds_since_ref = (t - tref).total_seconds()
        f.write(f"{seconds_since_ref:.1f} {precip_ts[i]:.4f}\n")

print(f"  ✓ Created {precip_path.name} ({len(times)} timesteps)")
print(f"    Peak: {precip_ts.max():.1f} mm/hr")
print(f"    Total: {precip_ts.sum():.0f} mm")

# ---- 2. Write discharge source files (if any) ----
if 'sources' in dir() and sources:
    src_path = model_path / "sfincs.src"
    with open(src_path, 'w') as f:
        for s in sources:
            f.write(f"{s['x']:.6f} {s['y']:.6f}\n")
    print(f"  ✓ Created {src_path.name} ({len(sources)} source points)")

    dis_path = model_path / "sfincs.dis"
    with open(dis_path, 'w') as f:
        f.write("0.0\n")
        for s in sources:
            f.write(f"{s['q']:.2f}\n")
        total_seconds = (end_date - start_date).total_seconds()
        f.write(f"{total_seconds:.1f}\n")
        for s in sources:
            f.write(f"{s['q']:.2f}\n")
    print(f"  ✓ Created {dis_path.name}")

# ---- 3. Update sfincs.inp ----
inp_path = model_path / "sfincs.inp"
with open(inp_path) as f:
    lines = f.readlines()

tref_str   = start_date.strftime("%Y%m%d %H%M%S")
tstart_str = start_date.strftime("%Y%m%d %H%M%S")
tstop_str  = end_date.strftime("%Y%m%d %H%M%S")

updates = {
    'tref': tref_str,
    'tstart': tstart_str,
    'tstop': tstop_str,
    'dtout': f"{dtout:.0f}",
    'precipfile': 'sfincs.precip',  # ASCII format instead of NetCDF
}

# Remove old precip2Dfile reference if it exists
remove_keys = {'precip2Dfile'}

if 'sources' in dir() and sources:
    updates['srcfile'] = 'sfincs.src'
    updates['disfile'] = 'sfincs.dis'

updated_keys = set()
new_lines = []
for line in lines:
    stripped = line.strip()
    matched = False
    if '=' in stripped:
        key = stripped.split('=', 1)[0].strip()
        if key in remove_keys:
            matched = True  # skip this line entirely
        elif key in updates:
            new_lines.append(f"{key:<20s} = {updates[key]}\n")
            updated_keys.add(key)
            matched = True
    if not matched:
        new_lines.append(line)

for key, val in updates.items():
    if key not in updated_keys:
        new_lines.append(f"{key:<20s} = {val}\n")

with open(inp_path, 'w') as f:
    f.writelines(new_lines)

print(f"\n  ✓ Updated sfincs.inp")
print(f"    tref       = {tref_str}")
print(f"    tstart     = {tstart_str}")
print(f"    tstop      = {tstop_str}")
print(f"    dtout      = {dtout:.0f} s")
print(f"    precipfile = sfincs.precip")

print(f"\n{'=' * 60}")
print("✓ Forcing setup complete! Ready to run SFINCS.")

### 4.4 Verify the Updated Input File

Let's inspect the updated `sfincs.inp` to confirm the forcing settings are correctly configured.

In [ ]:
with open(Path(f"./sfincs_models/{model_name}") / "sfincs.inp") as f:
    print(f.read())

---

## Part 5: Exploring the Model Structure

### 5.1 Model Directory Organization

After the build and forcing configuration, your model directory contains the following files:

**Expected directory structure**:

```
my_own_sfincs_model/
├── sfincs.inp          # Main input file — references all other files
├── sfincs.dep          # Bed elevation / DEM (binary)
├── sfincs.msk          # Active cell mask (binary)
├── sfincs.man          # Manning's roughness coefficients
├── sfincs.scs          # SCS curve numbers for infiltration
├── sfincs.src          # Discharge source point locations
├── sfincs.dis          # Discharge time series
├── precip_2d.nc        # 2D precipitation forcing (NetCDF)
├── sfincs_subgrid.nc   # Sub-grid lookup tables (NetCDF)
├── hydromt_data.yml    # Data catalog used during build
└── hydromt.log         # Build process log
```

### 5.2 Inspecting the Model with Python

HydroMT-SFINCS provides a Python API to load and inspect the built model. This is useful for verifying the setup, checking parameter values, and creating visualizations before running the simulation.

In [ ]:
from hydromt_sfincs import SfincsModel

mod = SfincsModel(f"./sfincs_models/{model_name}", mode="r")

print(f"Model region: {mod.region}")
print(f"Model CRS:    {mod.crs}")

### 5.3 Visualizing the Model Domain

Let's create a basemap visualization to verify that the model domain, elevation, and boundary conditions look correct.

In [ ]:
import os
import matplotlib.pyplot as plt
from hydromt_sfincs import SfincsModel

# List model files
model_path = f"./sfincs_models/{model_name}"
print("Model files:")
for f in sorted(os.listdir(model_path)):
    size = os.path.getsize(os.path.join(model_path, f))
    print(f"  {f:30s} {size:>10,} bytes")

# Load and inspect
mod = SfincsModel(model_path, mode="r")
mod.read_grid()
mod.read_config()

print(f"\nGrid variables: {list(mod.grid.data_vars)}")
print(f"Grid dims: {dict(mod.grid.dims)}")
print(f"CRS: {mod.crs}")

# Check dep values
if "dep" in mod.grid:
    dep = mod.grid["dep"]
    print(f"\nElevation stats:")
    print(f"  min:  {float(dep.min()):.2f}")
    print(f"  max:  {float(dep.max()):.2f}")
    print(f"  mean: {float(dep.mean()):.2f}")
    print(f"  NaN count: {int(dep.isnull().sum())}")

In [ ]:
def plot_sfincs_model(mod, model_name=None):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    mod.grid["dep"].plot(ax=axes[0], cmap="terrain")
    axes[0].set_title("Elevation [m]")
    axes[0].set_aspect("equal")
    
    mod.grid["msk"].plot(ax=axes[1], cmap="Blues")
    axes[1].set_title("Active Mask")
    axes[1].set_aspect("equal")
    
    mod.grid["manning"].plot(ax=axes[2], cmap="YlGn")
    axes[2].set_title("Manning Roughness")
    axes[2].set_aspect("equal")
    
    if model_name:
        fig.suptitle(f"SFINCS Model: {model_name}", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
    return fig, axes

plot_sfincs_model(mod, model_name)

**What to check**:
- Does the **model domain** cover your area of interest?
- Is the **elevation** reasonable for the terrain?
- Are **discharge source points** placed at the correct river locations?
- Does the **grid resolution** capture the key terrain features?

If something looks wrong, adjust the region or configuration and re-run the build.

### 5.4 Visualize the Forcing Data

Let's plot the forcing data to confirm the precipitation and discharge inputs look correct.

---

## Part 6: Running the Model

Now that the model is built and the forcing is configured, let's run SFINCS. We will:
1. Point to the folder where the SFINCS executable lives
2. Create a batch script using Python
3. Execute the model and stream the log output live

### 6.1 Set the Path to the SFINCS Executable

Enter the path to the folder containing `sfincs.exe`. On the training server, SFINCS is pre-installed. On your own machine, this would be the folder where you downloaded the executable from https://download.deltares.nl/sfincs.

**Tip**: The executable is also available in the software folder on the USB stick.

In [ ]:
# Set the path to the folder containing the SFINCS executable
sfincs_exe_folder = input("Path to folder containing sfincs.exe [/usr/local/bin]: ") or "/usr/local/bin"

# Verify the executable exists
sfincs_exe = Path(sfincs_exe_folder) / "sfincs" if os.name != "nt" else Path(sfincs_exe_folder) / "sfincs.exe"

if sfincs_exe.exists():
    print(f"✓ Found SFINCS executable: {sfincs_exe}")
else:
    print(f"⚠️ Executable not found at: {sfincs_exe}")
    print(f"  Available files in {sfincs_exe_folder}:")
    try:
        for f in Path(sfincs_exe_folder).iterdir():
            print(f"    {f.name}")
    except FileNotFoundError:
        print(f"  Folder does not exist!")
        # C:\backstop\Capacity-Building-ICPAC-for-climate-services\Session1\software\SFINCS

### 6.2 Create the Batch Script

We use Python to generate a `run.bat` (Windows) or `run.sh` (Linux) script inside the model directory. This script calls the SFINCS executable and pipes the output to a log file.

In [ ]:
import platform

model_path = Path(f"./sfincs_models/{model_name}")

if platform.system() == "Windows":
    # Create run.bat for Windows
    bat_content = f'call "{sfincs_exe.resolve()}" > sfincs_log.txt 2>&1\n'
    script_path = model_path / "run.bat"
    script_path.write_text(bat_content)
    print(f"✓ Created {script_path}")
    print(f"  Contents:")
    print(f"  {bat_content.strip()}")
else:
    # Create run.sh for Linux
    sh_content = f'#!/bin/bash\n"{sfincs_exe.resolve()}" > sfincs_log.txt 2>&1\n'
    script_path = model_path / "run.sh"
    script_path.write_text(sh_content)
    os.chmod(script_path, 0o755)
    print(f"✓ Created {script_path}")
    print(f"  Contents:")
    print(f"  {sh_content.strip()}")

### 6.3 Run the Model and Stream the Log

Now we execute SFINCS from Python. The output is streamed live into the notebook so you can follow the simulation progress in real time.

⚠️ **This can take up to 15 minutes depending on the model size and your hardware.**

In [ ]:
import subprocess

model_path = Path(f"./sfincs_models/{model_name}")
log_path = model_path / "sfincs_log.txt"

# Remove old log file if it exists
if log_path.exists():
    log_path.unlink()

print(f"Starting SFINCS in: {model_path}")
print(f"Executable: {sfincs_exe.resolve()}")
print("=" * 60)

# Start the SFINCS process
process = subprocess.Popen(
    [str(sfincs_exe.resolve())],
    cwd=str(model_path),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

# Stream output live
for line in process.stdout:
    print(line, end="")

# Wait for the process to finish
return_code = process.wait()

print("=" * 60)
if return_code == 0:
    print(f"✓ SFINCS finished successfully (exit code {return_code})")
else:
    print(f"⚠️ SFINCS finished with exit code {return_code}")

if log_path.exists():
    print(f"\nLog file saved to: {log_path}")

---

## Part 6.4: Visualize SFINCS Output

Now that the simulation has completed, let's load the output and visualize the flood results.

SFINCS produces two main output files:
- **`sfincs_map.nc`**: Gridded output (water levels, water depths) over the entire domain
- **`sfincs_his.nc`**: Time series at observation points (if configured)

We'll plot the **maximum water depth** on a satellite background to see the flood extent.

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from pathlib import Path

model_path = Path(f"./sfincs_models/{model_name}")

# Load the SFINCS map output
map_file = model_path / "sfincs_map.nc"

if map_file.exists():
    ds_map = xr.open_dataset(map_file)
    print("SFINCS map output loaded successfully!")
    print(f"Variables: {list(ds_map.data_vars)}")
    print(f"Time steps: {len(ds_map.timemax) if 'timemax' in ds_map.dims else 'N/A'}")
    print(f"\nDataset overview:")
    print(ds_map)
else:
    print(f"Output file not found: {map_file}")
    print("Make sure the SFINCS simulation completed successfully.")

In [ ]:
from hydromt_sfincs import SfincsModel

# Re-read model to get grid info
mod = SfincsModel(str(model_path), mode="r")
mod.read_grid()
mod.read_results()

# Get maximum water depth
# SFINCS outputs water levels (zsmax), we subtract elevation to get depth
if hasattr(mod, 'results') and mod.results is not None and 'zsmax' in mod.results:
    zsmax = mod.results['zsmax']
    dep = mod.grid['dep']
    hmax = zsmax - dep
    hmax = hmax.where(hmax > 0.01)  # mask very small depths
    print(f"Maximum water depth statistics:")
    print(f"  Max depth:  {float(hmax.max()):.2f} m")
    print(f"  Mean depth: {float(hmax.mean()):.2f} m")
    flooded_cells = int((hmax > 0.01).sum())
    total_cells = int((mod.grid['msk'] >= 1).sum())
    print(f"  Flooded cells: {flooded_cells} / {total_cells} ({100*flooded_cells/total_cells:.1f}%)")
elif 'zsmax' in ds_map:
    # Fallback: compute from raw NetCDF
    zsmax = ds_map['zsmax'].isel(timemax=0) if 'timemax' in ds_map['zsmax'].dims else ds_map['zsmax']
    dep = mod.grid['dep']
    hmax = zsmax - dep
    hmax = hmax.where(hmax > 0.01)
    print(f"Maximum water depth statistics:")
    print(f"  Max depth:  {float(hmax.max()):.2f} m")
    print(f"  Mean depth: {float(hmax.mean()):.2f} m")
else:
    print("Could not find water level output (zsmax).")
    print(f"Available variables: {list(ds_map.data_vars)}")

### 6.4.1 Maximum Flood Depth Map (with satellite background)

This plot overlays the simulated maximum water depth on a satellite image, making it easy to identify which areas are flooded.

In [ ]:
# Install contextily if not available (for satellite tile backgrounds)
try:
    import contextily as ctx
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'contextily', '--break-system-packages', '-q'])
    import contextily as ctx

from pyproj import Transformer
from matplotlib.colors import ListedColormap, BoundaryNorm
import numpy as np
import matplotlib.pyplot as plt

# ---- Collapse time dimension if present ----
if hasattr(hmax, 'dims') and 'time' in hmax.dims:
    hmax = hmax.max(dim='time')
elif hmax.ndim == 3:
    hmax = np.nanmax(hmax, axis=0)

# ---- Compute real-world UTM coordinates from rotated grid ----
# Read grid parameters from sfincs.inp
inp_path = Path(f"./sfincs_models/{model_name}/sfincs.inp")
inp_params = {}
with open(inp_path) as f:
    for line in f:
        if "=" in line:
            key, val = line.split("=", 1)
            inp_params[key.strip()] = val.strip()

x0 = float(inp_params['x0'])
y0 = float(inp_params['y0'])
dx = float(inp_params['dx'])
dy = float(inp_params['dy'])
mmax = int(inp_params['mmax'])
nmax = int(inp_params['nmax'])
rotation = float(inp_params.get('rotation', 0))

# Create local grid coordinates
cols = np.arange(mmax) * dx
rows = np.arange(nmax) * dy

# Apply rotation to get real UTM coordinates
rot_rad = np.radians(rotation)
cc, rr = np.meshgrid(cols, rows)
xx_utm = x0 + cc * np.cos(rot_rad) - rr * np.sin(rot_rad)
yy_utm = y0 + cc * np.sin(rot_rad) + rr * np.cos(rot_rad)

# ---- Reproject to Web Mercator (EPSG:3857) for satellite tiles ----
transformer_to_wm = Transformer.from_crs(f"EPSG:{inp_params['epsg']}", "EPSG:3857", always_xy=True)
transformer_to_ll = Transformer.from_crs(f"EPSG:{inp_params['epsg']}", "EPSG:4326", always_xy=True)

xx_wm, yy_wm = transformer_to_wm.transform(xx_utm, yy_utm)

# Get extent in Web Mercator (use actual corners, not just min/max)
xmin_wm, xmax_wm = xx_wm.min(), xx_wm.max()
ymin_wm, ymax_wm = yy_wm.min(), yy_wm.max()

# Get extent in lat/lon for display
xx_ll, yy_ll = transformer_to_ll.transform(xx_utm, yy_utm)

# ---- Create the figure ----
fig, ax = plt.subplots(figsize=(14, 10))

# Add some padding (5%)
pad_x = (xmax_wm - xmin_wm) * 0.05
pad_y = (ymax_wm - ymin_wm) * 0.05
ax.set_xlim(xmin_wm - pad_x, xmax_wm + pad_x)
ax.set_ylim(ymin_wm - pad_y, ymax_wm + pad_y)
ctx.add_basemap(ax, source=ctx.providers.Esri.WorldImagery, zoom='auto')

# ---- Overlay flood depth ----
levels = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
colors = ['#ffffcc', '#c7e9b4', '#7fcdbb', '#41b6c4', '#1d91c0', '#225ea8', '#0c2c84']
cmap = ListedColormap(colors)
norm = BoundaryNorm(levels, cmap.N)

# Get 2D flood depth data
hmax_data = hmax.values if hasattr(hmax, 'values') else hmax

# Plot flood depth on the correctly positioned grid
im = ax.pcolormesh(xx_wm, yy_wm, hmax_data, cmap=cmap, norm=norm, alpha=0.7, shading='auto')

# Colorbar
cbar = plt.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Maximum Water Depth [m]', fontsize=12)

# Title and labels
ax.set_title(f'SFINCS Flood Simulation: {model_name}\n'
             f'Maximum Water Depth ({start_date.strftime("%Y-%m-%d")} to {end_date.strftime("%Y-%m-%d")})',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Easting [m]')
ax.set_ylabel('Northing [m]')

plt.tight_layout()
plt.show()

print(f"\nModel extent (lat/lon): [{yy_ll.min():.4f}, {xx_ll.min():.4f}] to [{yy_ll.max():.4f}, {xx_ll.max():.4f}]")

### 6.4.2 Interactive Flood Map

This interactive map lets you zoom in and explore the flood results with a satellite background. Hover over flooded areas to see the water depth.

In [ ]:
# Install Pillow if not available
try:
    from PIL import Image
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'Pillow', '--break-system-packages', '-q'])
    from PIL import Image

import folium
from folium.raster_layers import ImageOverlay
from io import BytesIO
import base64
import numpy as np
from pyproj import Transformer
from scipy.interpolate import griddata

# ---- Read grid parameters (same as satellite plot) ----
inp_path = Path(f"./sfincs_models/{model_name}/sfincs.inp")
inp_params = {}
with open(inp_path) as f:
    for line in f:
        if "=" in line:
            key, val = line.split("=", 1)
            inp_params[key.strip()] = val.strip()

x0 = float(inp_params['x0'])
y0 = float(inp_params['y0'])
dx = float(inp_params['dx'])
dy = float(inp_params['dy'])
mmax = int(inp_params['mmax'])
nmax = int(inp_params['nmax'])
rotation = float(inp_params.get('rotation', 0))
epsg = inp_params['epsg']

# Create rotated UTM coordinates
cols = np.arange(mmax) * dx
rows = np.arange(nmax) * dy
rot_rad = np.radians(rotation)
cc, rr = np.meshgrid(cols, rows)
xx_utm = x0 + cc * np.cos(rot_rad) - rr * np.sin(rot_rad)
yy_utm = y0 + cc * np.sin(rot_rad) + rr * np.cos(rot_rad)

# Transform to lat/lon
transformer = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)
xx_ll, yy_ll = transformer.transform(xx_utm, yy_utm)

# ---- Regrid flood data to regular lat/lon grid ----
hmax_np = np.nan_to_num(hmax.values if hasattr(hmax, 'values') else hmax, nan=0.0)

# Flatten the rotated coordinates and data
points = np.column_stack([xx_ll.ravel(), yy_ll.ravel()])
values = hmax_np.ravel()

# Create a regular lat/lon grid
lon_min, lon_max = xx_ll.min(), xx_ll.max()
lat_min, lat_max = yy_ll.min(), yy_ll.max()
n_pixels = 500  # image resolution
lon_reg = np.linspace(lon_min, lon_max, n_pixels)
lat_reg = np.linspace(lat_min, lat_max, n_pixels)
lon_grid, lat_grid = np.meshgrid(lon_reg, lat_reg)

# Interpolate flood data onto regular grid
hmax_reg = griddata(points, values, (lon_grid, lat_grid), method='linear', fill_value=0.0)

# ---- Create color-mapped RGBA image ----
levels = [0.01, 0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
colors_rgba = [
    [0.0, 0.0, 0.0, 0.0],      # transparent (no flood)
    [1.0, 1.0, 0.8, 0.6],       # light yellow
    [0.78, 0.91, 0.71, 0.6],    # light green
    [0.50, 0.80, 0.73, 0.7],    # teal
    [0.25, 0.71, 0.77, 0.7],    # blue-teal
    [0.11, 0.57, 0.75, 0.8],    # medium blue
    [0.13, 0.37, 0.66, 0.8],    # dark blue
    [0.05, 0.17, 0.52, 0.9],    # very dark blue
]

# Dont show very low values
levels = [0.05, 0.1, 0.25, 0.5, 1.0, 2.0, 5.0]
colors_rgba = [
    [0.0, 0.0, 0.0, 0.0],      # transparent (below 0.05)
    [0.78, 0.91, 0.71, 0.6],    # light green        ← 0.05–0.10
    [0.50, 0.80, 0.73, 0.7],    # teal               ← 0.10–0.25
    [0.25, 0.71, 0.77, 0.7],    # blue-teal          ← 0.25–0.50
    [0.11, 0.57, 0.75, 0.8],    # medium blue        ← 0.50–1.00
    [0.13, 0.37, 0.66, 0.8],    # dark blue          ← 1.00–2.00
    [0.05, 0.17, 0.52, 0.9],    # very dark blue     ← > 2.00
]

rgba = np.zeros((*hmax_reg.shape, 4), dtype=np.float32)
for i in range(len(levels)):
    mask = hmax_reg >= levels[i]
    rgba[mask] = colors_rgba[i + 1] if i + 1 < len(colors_rgba) else colors_rgba[-1]

# Flip vertically (image origin is top-left, lat increases upward)
rgba_flipped = np.flipud(rgba)

# Convert to PNG
img = Image.fromarray((rgba_flipped * 255).astype(np.uint8), 'RGBA')
buffer = BytesIO()
img.save(buffer, format='PNG')
buffer.seek(0)

# ---- Create Folium map ----
center_lat = (lat_min + lat_max) / 2
center_lon = (lon_min + lon_max) / 2

m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=13,
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='Esri World Imagery',
)

folium.TileLayer('openstreetmap', name='OpenStreetMap').add_to(m)

# Overlay flood image at correct lat/lon bounds
img_base64 = base64.b64encode(buffer.getvalue()).decode()
img_url = f'data:image/png;base64,{img_base64}'

ImageOverlay(
    image=img_url,
    bounds=[[lat_min, lon_min], [lat_max, lon_max]],
    opacity=0.7,
    name='Max Water Depth',
).add_to(m)

folium.LayerControl().add_to(m)

# Add legend
legend_html = '''
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background-color: white; padding: 10px; border-radius: 5px;
     border: 2px solid grey; font-size: 12px;">
  <b>Max Water Depth [m]</b><br>
  <i style="background:#ffffcc; width:18px; height:12px; display:inline-block;"></i> 0.01 - 0.05<br>
  <i style="background:#c7e9b4; width:18px; height:12px; display:inline-block;"></i> 0.05 - 0.10<br>
  <i style="background:#7fcdbb; width:18px; height:12px; display:inline-block;"></i> 0.10 - 0.25<br>
  <i style="background:#41b6c4; width:18px; height:12px; display:inline-block;"></i> 0.25 - 0.50<br>
  <i style="background:#1d91c0; width:18px; height:12px; display:inline-block;"></i> 0.50 - 1.00<br>
  <i style="background:#225ea8; width:18px; height:12px; display:inline-block;"></i> 1.00 - 2.00<br>
  <i style="background:#0c2c84; width:18px; height:12px; display:inline-block;"></i> > 2.00<br>
</div>
'''
m.get_root().html.add_child(folium.Element(legend_html))

print(f"Interactive flood map for model: {model_name}")
print(f"Center: {center_lat:.4f}°N, {center_lon:.4f}°E")
m

---

## Part 7: Best Practices and Troubleshooting

### 7.1 Common Issues and Solutions

**Problem**: Build fails with "No data found for region"
- **Solution**: Check your region coordinates are correct (lon, lat order in GeoJSON)
- Verify the data catalog has coverage for your area
- Try a slightly larger bounding box

**Problem**: Grid is too coarse or too fine
- **Solution**: Adjust the `dx` and `dy` parameters in the build configuration YAML
- For urban areas, use 10–50 m resolution
- For regional studies, use 50–250 m resolution

**Problem**: Forcing files not recognized by SFINCS
- **Solution**: Verify that `sfincs.inp` references the correct file names
- Check that the precipitation NetCDF has the expected variable name (`Precipitation`)
- Ensure discharge time series cover the full simulation period

**Problem**: Unrealistically high or low water depths
- **Solution**: Check precipitation units (SFINCS expects mm/hr)
- Verify discharge magnitudes are realistic for the catchment size
- Inspect the DEM for artefacts or incorrect elevation values

### 7.2 Next Steps After Building

After verifying your model looks correct:
1. **Adjust forcing**: Replace static values with time-varying data from observations or forecasts
2. **Run the model**: Use the SFINCS executable (see Notebook 1 for details)
3. **Analyze results**: Load and visualize output flood maps and time series
4. **Iterate**: Adjust parameters, resolution, or domain as needed

---

## Summary and Next Steps

### What We've Learned

In this tutorial, you've learned the complete workflow for building your own SFINCS model:
1. ✓ Set up and verified the HydroMT-SFINCS installation
2. ✓ Selected a model region using an interactive map
3. ✓ Defined and saved the region as a GeoJSON polygon
4. ✓ Built a complete SFINCS model using HydroMT
5. ✓ Configured static forcing (precipitation and discharge)
6. ✓ Explored the model structure and visualized the setup

### Additional Resources

**Documentation**:
- HydroMT-SFINCS docs: https://deltares.github.io/hydromt_sfincs/
- SFINCS documentation: https://sfincs.readthedocs.io/
- HydroMT core: https://deltares.github.io/hydromt/
- SFINCS paper: https://www.sciencedirect.com/science/article/abs/pii/S0378383920304828